# Find a temperature with stable properties
Place a simulation CSV beside this notebook: one row per temperature, one column per property, and one consistent material/setup. Combine replicates before loading. Edit the four cells below. Requires NumPy and pandas.

The objective measures **change across temperature**, not convergence over simulation time. Interior temperatures are candidates; endpoints provide neighbors only. Bounds use the CSV units and are inclusive. Selecting among simulated temperatures needs no continuous optimizer or interpolation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# EDIT: CSV filename, column names, weights and bounds.
data_file = Path("simulation_data.csv")
temperature_column = "temperature_K"
temperature_units = "K"
properties = ["porosity", "density", "thermal_shrinkage"]
alpha = np.array([1.0, 1.0, 1.0])
conditions = {
    # "porosity": {"min": 0.35, "max": 0.60},
    # "thermal_shrinkage": {"max": 5.0},
}

print("Input:", data_file, "| Temperature units:", temperature_units)
print("Properties:", properties, "| Coefficients:", alpha)
print("Conditions:", conditions)

if not properties or len(set(properties)) != len(properties):
    raise ValueError("Choose distinct property columns.")
alpha = np.asarray(alpha, dtype=float)
if alpha.shape != (len(properties),) or not np.isfinite(alpha).all() or (alpha < 0).any() or not (alpha > 0).any():
    raise ValueError("Supply one finite nonnegative weight per property, with at least one positive weight.")
print("Weight mapping:", dict(zip(properties, alpha.tolist())))
for name, bounds in conditions.items():
    if not isinstance(bounds, dict) or not bounds or set(bounds) - {"min", "max"}:
        raise ValueError(f"Use min/max bounds for {name}.")
    if not all(np.isfinite(float(v)) for v in bounds.values()):
        raise ValueError(f"Nonfinite bound for {name}.")
    if bounds.get("min", -np.inf) > bounds.get("max", np.inf):
        raise ValueError(f"Reversed bounds for {name}.")
if temperature_column in properties or temperature_column in conditions:
    raise ValueError("Conditions and properties must be property columns, not temperature.")
df = pd.read_csv(data_file)
required = list(dict.fromkeys([temperature_column] + properties + list(conditions)))
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
df[required] = df[required].apply(pd.to_numeric, errors="raise")
if not np.isfinite(df[required].to_numpy(dtype=float)).all():
    raise ValueError("Selected temperatures, properties and constrained values must be finite; no automatic imputation.")
df = df.sort_values(temperature_column).reset_index(drop=True)
if len(df) < 3 or df[temperature_column].duplicated().any():
    raise ValueError("Supply at least three distinct temperatures, with one row per temperature.")


In [ ]:
# Centered secant derivative: g[t,n] = (x[t+1,n]-x[t-1,n]) / (theta[t+1]-theta[t-1]).
# This includes temperature spacing, including unequal spacing.
temperatures = df[temperature_column].to_numpy(dtype=float)
x = df[properties].to_numpy(dtype=float)
g = (x[2:] - x[:-2]) / (temperatures[2:] - temperatures[:-2])[:, None]
candidates = df.iloc[1:-1].copy().reset_index(drop=True)
slopes = pd.DataFrame(g, columns=properties, index=temperatures[1:-1])
slopes.index.name = temperature_column
print("Slopes (property units per temperature unit):")
display(slopes)


In [ ]:
# Sample SD across ALL supplied temperatures, including endpoints:
# sigma[n] = sqrt(sum_t((x[t,n] - mean[n])**2) / (T-1)).
# U[t] = sum_n(alpha[n] * (g[t,n] / sigma[n])**2).
sigma = x.std(axis=0, ddof=1)
normalized_slopes = np.divide(g, sigma, out=np.zeros_like(g), where=sigma > 0)
contributions = alpha * normalized_slopes**2
U = contributions.sum(axis=1)
if not np.isfinite(U).all():
    raise ValueError("Nonfinite objective; check input scale and numerical ranges.")
print("Sample standard deviations:", dict(zip(properties, sigma)))
print("Constant properties contribute zero:", [p for p, s in zip(properties, sigma) if s == 0])
print("Objective units: inverse squared temperature units. Weights are not automatically normalized.")
contribution_table = pd.DataFrame(contributions, columns=[f"contribution: {p}" for p in properties])
contribution_table.insert(0, temperature_column, temperatures[1:-1])
contribution_table["objective_U"] = U
display(contribution_table)


In [ ]:
# Apply inclusive property bounds at each candidate temperature, then minimize U.
feasible = np.ones(len(candidates), dtype=bool)
for name, bounds in conditions.items():
    values = candidates[name].to_numpy(dtype=float)
    if "min" in bounds:
        feasible &= values >= float(bounds["min"])
    if "max" in bounds:
        feasible &= values <= float(bounds["max"])
results = candidates[list(dict.fromkeys([temperature_column] + properties + list(conditions)))].copy()
results["objective_U"] = U
results["feasible"] = feasible
display(results)
if not feasible.any():
    best_temperatures = results.iloc[0:0].copy()
    print("No interior temperature satisfies all property bounds.")
else:
    minimum_U = U[feasible].min()
    best_temperatures = results.loc[feasible & (U == minimum_U)].copy()
    print("Minimum feasible objective:", minimum_U)
    print("Selected temperature(s), including exact ties:")
    display(best_temperatures)
